# 3. Combat

The densest rules in the engine, and the part a deck search leans on hardest —
in a creature format, combat *is* the game.

Four things happen in order: declare attackers, declare blockers, assign damage,
end of combat. Each has its own rules, and the keywords change all of them.

In [1]:
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "data" / "cards.duckdb").exists() and REPO != REPO.parent:
    REPO = REPO.parent

from manalab.cards.mana import parse_cost
from manalab.cards.model import Card
from manalab.cards.store import CardStore

store = CardStore(REPO / "data" / "cards.duckdb")


def creature(name, power, toughness, *keywords, cost="{1}{G}"):
    # A synthetic creature, so a demo shows a rule and not a card.
    return Card(name.lower().replace(" ", "-"), name, parse_cost(cost),
                "Creature - Test", power=str(power), toughness=str(toughness),
                keywords=frozenset(keywords))


FOREST = store.by_name("Forest")

In [2]:
from manalab.engine.zones import Zone


def show(state, title=""):
    # Print the whole visible game state. The workhorse of these notebooks.
    if title:
        print(f"--- {title}")
    print(f"    turn {state.turn}, active P{state.active_player}")
    for player in state.players:
        flags = " LOST" if player.has_lost else ""
        print(f"    P{player.index}: {player.life} life{flags}"
              f"   hand={len(state.hand(player.index))}"
              f"  library={len(state.library(player.index))}"
              f"  graveyard={len(state.graveyard(player.index))}")
        for obj in state.battlefield(player.index):
            marks = []
            if obj.tapped:
                marks.append("tapped")
            if obj.damage:
                marks.append(f"dmg{obj.damage}")
            if obj.deathtouched:
                marks.append("deathtouched")
            if obj.summoning_sick:
                marks.append("sick")
            if obj.attacking is not None:
                marks.append(f"attacking P{obj.attacking}")
            if obj.blocking is not None:
                marks.append("blocking")
            if obj.counters:
                marks.extend(f"{k}x{v}" for k, v in sorted(obj.counters.items()))
            body = f"{obj.power}/{obj.toughness}" if obj.is_creature else "land"
            suffix = f"  [{' '.join(marks)}]" if marks else ""
            print(f"        #{obj.id} {obj.card.name:<22} {body:<6}{suffix}")
    stack = state.zone(Zone.STACK)
    if stack:
        print(f"    stack (top last): {[o.card.name for o in stack]}")

In [3]:
from manalab.engine import combat, sba
from manalab.engine.state import GameState
from manalab.engine.zones import Zone


def battlefield(attackers, blockers):
    # P0 attacks P1 with everything; nothing is blocking yet.
    state = GameState.new([[], []], seed=0)
    state.active_player = 0
    for card in attackers:
        obj = state.create(card, owner=0, zone=Zone.BATTLEFIELD)
        obj.attacking = 1
    for card in blockers:
        state.create(card, owner=1, zone=Zone.BATTLEFIELD, controller=1)
    return state


def attackers_of(state):
    return [o for o in state.battlefield() if o.attacking is not None]


def block(state, blocker_index, attacker_index):
    blocker = state.battlefield(1)[blocker_index]
    attacker = attackers_of(state)[attacker_index]
    blocker.blocking = attacker.id
    attacker.blocked_by = attacker.blocked_by + (blocker.id,)


def strike(state):
    # Both damage passes plus state-based actions, exactly as the driver does.
    live = attackers_of(state)
    if combat.any_first_strike(live, state):
        for line in combat.combat_damage(state, live, first_strike_pass=True):
            print("      first strike:", line)
        sba.check(state)
        live = attackers_of(state)
    for line in combat.combat_damage(state, live, first_strike_pass=False):
        print("      damage:", line)
    sba.check(state)

## Who can attack

`can_attack` is a property of the object: a creature, untapped, not summoning
sick (unless it has haste), and not a Defender.

In [4]:
state = GameState.new([[], []], seed=0)
cases = {
    "plain, been here a turn": creature("Veteran", 2, 2),
    "just cast (sick)": creature("Fresh", 2, 2),
    "just cast, hasty": creature("Hasty", 2, 2, "Haste"),
    "a wall": creature("Wall", 0, 4, "Defender"),
}
for label, card in cases.items():
    obj = state.create(card, owner=0, zone=Zone.BATTLEFIELD)
    obj.summoning_sick = "just cast" in label
    print(f"  {label:<26} can_attack={obj.can_attack}")

tapped = state.create(creature("Tapped", 2, 2), owner=0, zone=Zone.BATTLEFIELD)
tapped.tapped = True
print(f"  {'already tapped':<26} can_attack={tapped.can_attack}")

  plain, been here a turn    can_attack=True
  just cast (sick)           can_attack=False
  just cast, hasty           can_attack=True
  a wall                     can_attack=False
  already tapped             can_attack=False


## Attacking taps you — unless you have vigilance

In [5]:
from manalab.engine import actions as A
from manalab.engine.agents import CurveAgent, PassingAgent
from manalab.engine.game import Game

state = GameState.new([[], []], seed=0)
state.active_player = 0
plain = state.create(creature("Plain", 3, 3), owner=0, zone=Zone.BATTLEFIELD)
vigilant = state.create(creature("Vigilant", 3, 3, "Vigilance"), owner=0,
                        zone=Zone.BATTLEFIELD)

game = Game(state, [CurveAgent(), PassingAgent()])
game._declare_attackers()
for obj in state.battlefield(0):
    print(f"  {obj.card.name:<10} attacking={obj.attacking is not None}  tapped={obj.tapped}")

  Plain      attacking=True  tapped=True
  Vigilant   attacking=True  tapped=False


## Blocking: evasion decides the pairings

`combat.can_block` answers one question — may *this* creature block *that* one.
Flying is the classic: a ground creature cannot block a flier, but Reach can.

In [6]:
FLYER = creature("Flyer", 2, 2, "Flying")
GROUND = creature("Ground", 3, 3)
SPIDER = creature("Spider", 2, 4, "Reach")

for blocker_card in (GROUND, SPIDER, FLYER):
    state = battlefield([FLYER], [blocker_card])
    ok = combat.can_block(state.battlefield(1)[0], attackers_of(state)[0])
    print(f"  {blocker_card.name:<8} blocks a flier? {ok}")

  Ground   blocks a flier? False
  Spider   blocks a flier? True
  Flyer    blocks a flier? True


### Menace is a property of the whole assignment

"Can't be blocked except by two or more creatures" is not about any one pairing,
so it cannot live in `can_block`. It is enforced once the full assignment is
known — which is why `legal_blocks` returns *complete* assignments.

In [7]:
SNEAK = creature("Sneak", 3, 3, "Menace")
state = battlefield([SNEAK], [GROUND, creature("Other", 2, 2)])
options, truncated = combat.legal_blocks(state, 1, [a.id for a in attackers_of(state)])

print("legal block assignments against a menace attacker:")
for option in options:
    names = [state.objects[b].card.name for b, _ in option]
    print(f"   {len(option)} blocker(s): {names or 'none'}")
print("\nblocking with exactly one is never offered.")

legal block assignments against a menace attacker:
   0 blocker(s): none
   2 blocker(s): ['Ground', 'Other']

blocking with exactly one is never offered.


### Multiple blockers, multiple attackers

Each untapped creature may block one attacker, and different blockers may block
different attackers. This was once limited to *one blocker in total*, which
quietly made "go as wide as possible" a dominant strategy for reasons that had
nothing to do with Magic.

In [8]:
state = battlefield([creature("A1", 2, 2), creature("A2", 3, 3)],
                    [creature("B1", 2, 2), creature("B2", 3, 3)])
ids = [a.id for a in attackers_of(state)]
options, truncated = combat.legal_blocks(state, 1, ids)

print(f"{len(options)} legal assignments with 2 attackers and 2 blockers:")
for option in options:
    pairs = [f"{state.objects[b].card.name}->{state.objects[a].card.name}"
             for b, a in option]
    print("   ", pairs or ["no blocks"])

9 legal assignments with 2 attackers and 2 blockers:
    ['no blocks']
    ['B1->A1']
    ['B1->A2']
    ['B2->A1']
    ['B2->A1', 'B1->A1']
    ['B2->A1', 'B1->A2']
    ['B2->A2']
    ['B2->A2', 'B1->A1']
    ['B2->A2', 'B1->A2']


In [9]:
# The space is (attackers+1)^blockers, so it is capped and says when it truncated.
wide = battlefield([creature(f"A{i}", 2, 2) for i in range(8)],
                   [creature(f"B{i}", 2, 2) for i in range(8)])
options, truncated = combat.legal_blocks(wide, 1, [a.id for a in attackers_of(wide)])
print(f"8 vs 8: {len(options)} options offered, truncated={truncated}")
print(f"cap is {combat.MAX_BLOCK_OPTIONS}; the full space would be 9**8 = {9**8:,}")

8 vs 8: 729 options offered, truncated=True
cap is 2048; the full space would be 9**8 = 43,046,721


## Damage

Three rules run the whole thing.

1. **Damage is simultaneous.** A creature that dies still dealt its damage.
2. **An attacker assigns lethal in order** down its blockers, and excess is
   wasted — unless it has trample.
3. **Deathtouch makes one point lethal**, which changes what "lethal" means in
   rule 2.

Start with the simplest case: a trade.

In [10]:
state = battlefield([creature("Bear", 2, 2)], [creature("Bear", 2, 2)])
block(state, 0, 0)
show(state, "before damage")
strike(state)
show(state, "after — both died, because damage is simultaneous")

--- before damage
    turn 0, active P0
    P0: 20 life   hand=0  library=0  graveyard=0
        #1 Bear                   2/2     [attacking P1]
    P1: 20 life   hand=0  library=0  graveyard=0
        #2 Bear                   2/2     [blocking]
--- after — both died, because damage is simultaneous
    turn 0, active P0
    P0: 20 life   hand=0  library=0  graveyard=1
    P1: 20 life   hand=0  library=0  graveyard=1


In [11]:
# Excess damage is wasted without trample...
state = battlefield([creature("Giant", 5, 5)], [creature("Bear", 2, 2)])
block(state, 0, 0)
strike(state)
print(f"  defender life: {state.players[1].life}   (no trample: 3 excess wasted)")

# ...and spills over with it.
state = battlefield([creature("Trampler", 5, 5, "Trample")], [creature("Bear", 2, 2)])
block(state, 0, 0)
strike(state)
print(f"  defender life: {state.players[1].life}   (trample: 2 lethal, 3 through)")

  defender life: 20   (no trample: 3 excess wasted)
      damage: Trampler tramples over for 3
  defender life: 17   (trample: 2 lethal, 3 through)


In [12]:
# Deathtouch: one point is lethal, so a 1/1 eats a 10/10...
state = battlefield([creature("Colossus", 10, 10)], [creature("Snake", 1, 1, "Deathtouch")])
block(state, 0, 0)
strike(state)
print("  after:", {p.index: [o.card.name for o in state.battlefield(p.index)]
                   for p in state.players})

# ...and deathtouch + trample only needs to assign 1 before trampling.
state = battlefield([creature("Wurm", 5, 5, "Trample", "Deathtouch")],
                    [creature("Giant", 5, 5)])
block(state, 0, 0)
strike(state)
print(f"  deathtouch+trample defender life: {state.players[1].life}  (1 lethal, 4 through)")

  after: {0: [], 1: []}
      damage: Wurm tramples over for 4
  deathtouch+trample defender life: 16  (1 lethal, 4 through)


In [13]:
# First strike runs as a separate pass, with state-based actions in between.
state = battlefield([creature("Knight", 2, 2, "First strike")], [creature("Bear", 2, 2)])
block(state, 0, 0)
strike(state)
print("  survivors:", {p.index: [o.card.name for o in state.battlefield(p.index)]
                       for p in state.players})
print("  the Knight lived: the Bear died before it could strike back")

  survivors: {0: ['Knight'], 1: []}
  the Knight lived: the Bear died before it could strike back


In [14]:
# Ganging up: lethal is assigned in order, so a 5/5 kills two 2/2s and not the third.
state = battlefield([creature("Giant", 5, 5)],
                    [creature("B1", 2, 2), creature("B2", 2, 2), creature("B3", 2, 2)])
for i in range(3):
    block(state, i, 0)
show(state, "three blockers on one attacker")
strike(state)
show(state, "after — 2+2 lethal, 1 spills into the third; 6 back kills the Giant")

--- three blockers on one attacker
    turn 0, active P0
    P0: 20 life   hand=0  library=0  graveyard=0
        #1 Giant                  5/5     [attacking P1]
    P1: 20 life   hand=0  library=0  graveyard=0
        #2 B1                     2/2     [blocking]
        #3 B2                     2/2     [blocking]
        #4 B3                     2/2     [blocking]
--- after — 2+2 lethal, 1 spills into the third; 6 back kills the Giant
    turn 0, active P0
    P0: 20 life   hand=0  library=0  graveyard=1
    P1: 20 life   hand=0  library=0  graveyard=2
        #4 B3                     2/2     [dmg1 blocking]


## Which keywords actually move a win rate

The unit tests prove each keyword does the right thing. This is the aggregate:
a 3/3 with one keyword, against a deck of plain 3/3s, 400 games each.

In [15]:
from manalab.cards.model import Decklist
from manalab.evaluate.match import GauntletDeck, MatchEvaluator

def body(*keywords):
    return creature("Test Beater", 3, 3, *keywords, cost="{2}{G}")

plain = GauntletDeck("plain", tuple([FOREST] * 24 + [body()] * 36))

for kws in [(), ("Vigilance",), ("Trample",), ("Lifelink",), ("Deathtouch",),
            ("Flying",), ("First strike",), ("Double strike",)]:
    ev = MatchEvaluator(gauntlet=(plain,),
                        cards={"Forest": FOREST, "Test Beater": body(*kws)})
    r = ev.evaluate(Decklist({"Forest": 24, "Test Beater": 36}), n_games=400, seed=1)
    lo, hi = r.ci95("win_rate")
    label = ", ".join(kws) or "(mirror)"
    print(f"  {label:<16} {r.metrics['win_rate']:.3f}   95% CI [{lo:.3f}, {hi:.3f}]")

  (mirror)         0.531   95% CI [0.493, 0.569]


  Vigilance        0.531   95% CI [0.494, 0.569]


  Trample          0.531   95% CI [0.493, 0.569]


  Lifelink         0.611   95% CI [0.580, 0.642]


  Deathtouch       0.677   95% CI [0.640, 0.715]


  Flying           0.745   95% CI [0.703, 0.787]


  First strike     0.935   95% CI [0.911, 0.959]


  Double strike    0.945   95% CI [0.923, 0.967]


The mirror sits near 0.500 with a confidence interval covering it, which is the
sanity check that matters: identical decks, balanced for the play/draw advantage,
must be a coin flip.

**Vigilance and Trample come out identical to the mirror** — same mean, same
interval, to three decimals. That is not a rounding coincidence: under common
random numbers these decks met exactly the same shuffles, and neither keyword
changed a single game. Both results are *correct*:

- **Trample** only pays when an attacker outsizes its blocker. Two 3/3s trading
  leave no excess to trample over — the hand-built example above showed it
  working when there *is* excess.
- **Vigilance** matters when you want to attack and still block. `CurveAgent`
  attacks with everything and both seats run the same pilot, so the untapped
  body never gets used.

A keyword reading as neutral is a statement about the *pilot and the matchup*,
not about the keyword.

## What the engine still does not do in combat

Stated plainly, because a search will find every one of these:

- **Damage assignment order is fixed** (declaration order), not chosen by the attacker.
- **No protection, hexproof, indestructible, banding, or damage prevention.**
- **First strike is two passes inside one step**, not a separate step with its own
  priority window. Nothing can be cast in the gap today, so the only observable
  difference is that state-based actions run between passes — which they do.

Next: spells that actually do something.